# Model Experimentation — ResNet50 vs HRNet × Frame Count × Bird

Compare DeepLabCut training performance across:

- **Architectures**: ResNet-50, HRNet-W32
- **Training frame counts**: 400, 800, 1400
- **Birds**: Miguel, DavidBowie, Endive

A fixed holdout of **200 frames** (per bird) is reserved with a separate seed so RMSE and training-time comparisons remain fair across all 18 runs (2 archs × 3 frame counts × 3 birds).

In [1]:
import os
import glob
import json
import logging
import random
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import sys
sys.path.insert(0, str(Path(".").resolve()))   # so DLCsupport.py is importable

import DLCsupport as dlcs
from DLCsupport import (
    as_posix_str,
    validate_path_exists,
    find_latest_snapshot,
    load_bodyparts,
    create_combined_project_if_missing,
    build_combined_dataset,
)

import deeplabcut
import xrommtools

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Seed        : {SEED}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"DeepLabCut  : {deeplabcut.__version__}")

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)
Seed        : 42
NumPy       : 1.23.5
Pandas      : 2.3.3
DeepLabCut  : 3.0.0rc13


In [ ]:
## 2. Global Constants
# Centralize seeds, frame budgets, architecture names, and safety toggles.

In [2]:
# Workspace root (one level above Code-Testing/)
ROOT = Path("..")

TASK         = "Canari"
EXPERIMENTER = "Tyler"

# Frame counts to evaluate — one combined project is created per (bird, n)
FRAME_COUNTS  = [200, 400, 800, 1400]

# Frames held out for evaluation. Built once per bird with HOLDOUT_SEED
# so the exact same 200 frames are used across every training comparison.
HOLDOUT_N    = 200
HOLDOUT_SEED = 999   # kept different from TRAIN_SEED on purpose
TRAIN_SEED   = SEED  # reproducible training-frame selection

# DLC PyTorch backbone names to compare
ARCHITECTURES = ["resnet_50"]

EPOCHS_SMOKE = 2
EPOCHS_FULL  = 200

# Safety toggles  — flip to True to actually run training / analysis
RUN_TRAINING      = False
RUN_VIDEO_ANALYSIS = False
USE_FULL_EPOCHS   = False

print("Global constants ready.")
print(f"  Frame counts  : {FRAME_COUNTS}")
print(f"  Architectures : {ARCHITECTURES}")
print(f"  Holdout N     : {HOLDOUT_N}  (seed {HOLDOUT_SEED})")
print(f"  Train seed    : {TRAIN_SEED}")
print(f"  Epochs        : {EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE} ({'FULL' if USE_FULL_EPOCHS else 'SMOKE'})")

Global constants ready.
  Frame counts  : [200, 400, 800, 1400]
  Architectures : ['resnet_50']
  Holdout N     : 200  (seed 999)
  Train seed    : 42
  Epochs        : 2 (SMOKE)


## 3. Per-Bird Configuration

Each bird gets its own cell with all necessary paths.
Update the date-stamped subfolder (e.g. `Canari-Tyler-2026-03-31`) if you create new single-camera projects on a different date.

In [8]:
# ── Miguel ─────────────────────────────────────────────────────────────────
MIGUEL_ROOT = ROOT / "DeepLabCut" / "Miguel"

MIGUEL_CAM1_CONFIG = (
    MIGUEL_ROOT / "TestData" / "Canari_cam1_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)
MIGUEL_CAM2_CONFIG = (
    MIGUEL_ROOT / "TestData" / "Canari_cam2_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)

# Root folder that will contain all Miguel combined-training sub-projects
MIGUEL_COMBINED_BASE = MIGUEL_ROOT / "TestData"

MIGUEL_DATA_PATH    = MIGUEL_ROOT / "trainingdata"
MIGUEL_DATASET_NAME = "Canari"

# Any existing .avi used only to satisfy DLC project-creation API
MIGUEL_DUMMY_VIDEO  = (
    MIGUEL_ROOT / "trainingdata" / "Miguel_20250816_F6"
    / "MiguelF6_20250816_Cam1_short.avi"
)

# Videos used for evaluation (not seen during training)
MIGUEL_TEST_VIDEO_CAM1 = (
    MIGUEL_ROOT / "newdata" / "Miguel20260401Trial6_Cam1"
    / "MiguelF6_20250816_Cam1_short.avi"
)
MIGUEL_TEST_VIDEO_CAM2 = (
    MIGUEL_ROOT / "newdata" / "Miguel20260401Trial6_Cam2"
    / "MiguelF6_20250816_Cam2_short.avi"
)

print("Miguel paths:")
print("  Cam1 config :", MIGUEL_CAM1_CONFIG)
print("  Cam2 config :", MIGUEL_CAM2_CONFIG)
print("  Data path   :", MIGUEL_DATA_PATH)
print("  Dummy video :", MIGUEL_DUMMY_VIDEO)
print("  Test cam1   :", MIGUEL_TEST_VIDEO_CAM1)
print("  Test cam2   :", MIGUEL_TEST_VIDEO_CAM2)

Miguel paths:
  Cam1 config : ..\DeepLabCut\Miguel\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\Miguel\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\Miguel\trainingdata
  Dummy video : ..\DeepLabCut\Miguel\trainingdata\Miguel_20250816_F6\MiguelF6_20250816_Cam1_short.avi
  Test cam1   : ..\DeepLabCut\Miguel\newdata\Miguel20260401Trial6_Cam1\MiguelF6_20250816_Cam1_short.avi
  Test cam2   : ..\DeepLabCut\Miguel\newdata\Miguel20260401Trial6_Cam2\MiguelF6_20250816_Cam2_short.avi


In [3]:
# ── DavidBowie ─────────────────────────────────────────────────────────────
# NOTE: Mirror Miguel's folder tree under DeepLabCut/DavidBowie/.
#       Update the date-stamped subfolder once the single-camera projects exist.
DAVIDBOWIE_ROOT = ROOT / "DeepLabCut" / "DavidBowie"

DAVIDBOWIE_CAM1_CONFIG = (
    DAVIDBOWIE_ROOT / "TestData" / "Canari_cam1_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)
DAVIDBOWIE_CAM2_CONFIG = (
    DAVIDBOWIE_ROOT / "TestData" / "Canari_cam2_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)

DAVIDBOWIE_COMBINED_BASE = DAVIDBOWIE_ROOT / "TestData"

DAVIDBOWIE_DATA_PATH    = DAVIDBOWIE_ROOT / "trainingdata"
DAVIDBOWIE_DATASET_NAME = "Canari"

# Replace with the actual training video once it exists
DAVIDBOWIE_DUMMY_VIDEO  = (
    DAVIDBOWIE_ROOT / "trainingdata" / "DavidBowie_20250828_F17"
    / "DB17_Cam1.avi"
)

DAVIDBOWIE_TEST_VIDEO_CAM1 = (
    DAVIDBOWIE_ROOT / "newdata" / "DavidBowie_20250828_F17"
    / "DB17_Cam1.avi"
)
DAVIDBOWIE_TEST_VIDEO_CAM2 = (
    DAVIDBOWIE_ROOT / "newdata" / "DavidBowie_20250828_F17"
    / "DB17_Cam2.avi"
)

print("DavidBowie paths:")
print("  Cam1 config :", DAVIDBOWIE_CAM1_CONFIG)
print("  Cam2 config :", DAVIDBOWIE_CAM2_CONFIG)
print("  Data path   :", DAVIDBOWIE_DATA_PATH)
print("  Dummy video :", DAVIDBOWIE_DUMMY_VIDEO)
print("  Test cam1   :", DAVIDBOWIE_TEST_VIDEO_CAM1)
print("  Test cam2   :", DAVIDBOWIE_TEST_VIDEO_CAM2)

DavidBowie paths:
  Cam1 config : ..\DeepLabCut\DavidBowie\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\DavidBowie\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\DavidBowie\trainingdata
  Dummy video : ..\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\DB17_Cam1.avi
  Test cam1   : ..\DeepLabCut\DavidBowie\newdata\DavidBowie_20250828_F17\DB17_Cam1.avi
  Test cam2   : ..\DeepLabCut\DavidBowie\newdata\DavidBowie_20250828_F17\DB17_Cam2.avi


In [4]:
# ── Endive ──────────────────────────────────────────────────────────────────
# NOTE: Mirror Miguel's folder tree under DeepLabCut/Endive/.
#       Update the date-stamped subfolder once the single-camera projects exist.
ENDIVE_ROOT = ROOT / "DeepLabCut" / "Endive"

ENDIVE_CAM1_CONFIG = (
    ENDIVE_ROOT / "TestData" / "Canari_cam1_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)
ENDIVE_CAM2_CONFIG = (
    ENDIVE_ROOT / "TestData" / "Canari_cam2_training"
    / "Canari-Tyler-2026-03-31" / "config.yaml"
)

ENDIVE_COMBINED_BASE = ENDIVE_ROOT / "TestData"

ENDIVE_DATA_PATH    = ENDIVE_ROOT / "trainingdata"
ENDIVE_DATASET_NAME = "Canari"

# Replace with the actual training video once it exists
ENDIVE_DUMMY_VIDEO  = (
    ENDIVE_ROOT / "trainingdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam1_short.avi"
)


ENDIVE_TEST_VIDEO_CAM1 = (
    ENDIVE_ROOT / "newdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam1_short.avi"
)
ENDIVE_TEST_VIDEO_CAM2 = (
    ENDIVE_ROOT / "newdata" / "Endive_20251606_F32"
    / "EndiveF42_20251606_Cam2_short.avi"
)

print("Endive paths:")
print("  Cam1 config :", ENDIVE_CAM1_CONFIG)
print("  Cam2 config :", ENDIVE_CAM2_CONFIG)
print("  Data path   :", ENDIVE_DATA_PATH)
print("  Dummy video :", ENDIVE_DUMMY_VIDEO)
print("  Test cam1   :", ENDIVE_TEST_VIDEO_CAM1)
print("  Test cam2   :", ENDIVE_TEST_VIDEO_CAM2)

Endive paths:
  Cam1 config : ..\DeepLabCut\Endive\TestData\Canari_cam1_training\Canari-Tyler-2026-03-31\config.yaml
  Cam2 config : ..\DeepLabCut\Endive\TestData\Canari_cam2_training\Canari-Tyler-2026-03-31\config.yaml
  Data path   : ..\DeepLabCut\Endive\trainingdata
  Dummy video : ..\DeepLabCut\Endive\trainingdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi
  Test cam1   : ..\DeepLabCut\Endive\newdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi
  Test cam2   : ..\DeepLabCut\Endive\newdata\Endive_20251606_F32\EndiveF42_20251606_Cam2_short.avi


## 4. Bird Registry
Collect all per-bird variables into one dict so the rest of the notebook can iterate without repeating code.

In [5]:
BIRDS = {

    "DavidBowie": {
        "cam1_config"    : DAVIDBOWIE_CAM1_CONFIG,
        "cam2_config"    : DAVIDBOWIE_CAM2_CONFIG,
        "combined_base"  : DAVIDBOWIE_COMBINED_BASE,
        "data_path"      : DAVIDBOWIE_DATA_PATH,
        "dataset_name"   : DAVIDBOWIE_DATASET_NAME,
        "dummy_video"    : DAVIDBOWIE_DUMMY_VIDEO,
        "test_videos"    : [DAVIDBOWIE_TEST_VIDEO_CAM1, DAVIDBOWIE_TEST_VIDEO_CAM2],
    },
    "Endive": {
        "cam1_config"    : ENDIVE_CAM1_CONFIG,
        "cam2_config"    : ENDIVE_CAM2_CONFIG,
        "combined_base"  : ENDIVE_COMBINED_BASE,
        "data_path"      : ENDIVE_DATA_PATH,
        "dataset_name"   : ENDIVE_DATASET_NAME,
        "dummy_video"    : ENDIVE_DUMMY_VIDEO,
        "test_videos"    : [ENDIVE_TEST_VIDEO_CAM1, ENDIVE_TEST_VIDEO_CAM2],
    },
}

print(f"Registered {len(BIRDS)} birds: {list(BIRDS)}")

Registered 2 birds: ['DavidBowie', 'Endive']


## 5. Create Combined Projects

One combined DLC project is created for **each (bird, frame-count) pair** — 9 training projects total.
A separate **holdout project** (200 frames) is also created per bird so evaluation frames are never in any training set.

All 12 projects are stored in `combined_configs`, keyed by `(bird_name, nframes)` or `(bird_name, "holdout")`.

In [ ]:
# combined_configs[(bird_name, nframes)]   -> Path to config.yaml  (training projects)
# combined_configs[(bird_name, "holdout")] -> Path to config.yaml  (evaluation project)
combined_configs: dict = {}

for bird_name, bird_cfg in BIRDS.items():
    base = bird_cfg["combined_base"]   # e.g. .../Miguel/TestData/
    dummy = bird_cfg["dummy_video"]


    # ── Training projects — one per frame count ───────────────────────────────
    for n in FRAME_COUNTS:
        cfg_path = create_combined_project_if_missing(
            task=TASK,
            experimenter=EXPERIMENTER,
            combined_project_root=base / f"Canari_combined_n{n}",
            dummy_video=dummy,
        )
        combined_configs[(bird_name, n)] = cfg_path

    print(f"{bird_name}: {len(FRAME_COUNTS)} training + 1 holdout projects ready.")

print(f"\nTotal combined configs registered: {len(combined_configs)}")
for key, path in combined_configs.items():
    print(f"  {key[0]:12s}  {'holdout' if key[1] == 'holdout' else f'n={key[1]:>4}'}  ->  {path}")

#FIXME: update to include edit config command from deeplabcut, input config path and disctionary {bodyparts: ["bodypart1", "bodypart2", ...]}

Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\labeled-data"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\training-datasets"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.


INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\config.yaml


C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\DB17_cam1.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\videos\DB17_cam1.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\videos\DB17_cam1.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\config.yaml"

A new project with name Canari-Tyler-2026-04-08 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\config.yaml


C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\trainingdata\DavidBowie_20250828_F17\DB17_cam1.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\videos\DB17_cam1.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\videos\DB17_cam1.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\config.yaml"

A new project with name Canari-Tyler-2026-04-08 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have

...

C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\trainingdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\config.yaml"

A new project with name Canari-Tyler-2026-04-08 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n400 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to 

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\config.yaml


C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\trainingdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\config.yaml"

A new project with name Canari-Tyler-2026-04-08 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to 

INFO | Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\config.yaml


C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\trainingdata\Endive_20251606_F32\EndiveF42_20251606_Cam1_short.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\videos\EndiveF42_20251606_Cam1_short.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\config.yaml"

A new project with name Canari-Tyler-2026-04-08 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt

## 6. Build Combined Datasets

Build **9 training datasets** (3 birds × 3 frame counts) plus **3 holdout datasets** (one per bird, 200 frames each).

**Holdout strategy**:  
- Holdout frames are sampled with `HOLDOUT_SEED = 999`.  
- Training frames are sampled with `TRAIN_SEED = 42`.  
- Because the seeds differ, overlap between holdout and training pools is negligible. The same holdout frames will be produced identically every time this cell is re-run.

In [7]:
dataset_build_log = []   # records what was (or would be) built

for bird_name, bird_cfg in BIRDS.items():

    for n in FRAME_COUNTS:
        train_cfg = combined_configs[(bird_name, n)]
        print(f"[{bird_name}] Building training dataset (n={n} frames, seed={TRAIN_SEED}) ...")
        build_combined_dataset(
            combined_config     = train_cfg,
            data_path           = bird_cfg["data_path"],
            dataset_name        = bird_cfg["dataset_name"],
            experimenter        = EXPERIMENTER,
            nframes             = n,
            frame_selection_seed= TRAIN_SEED,
        )
        dataset_build_log.append({"bird": bird_name, "split": "train", "nframes": n, "config": str(train_cfg)})

print(f"\n{'─'*60}")
print(f"Dataset builds complete: {len([r for r in dataset_build_log if r['split']=='train'])} training  "
      f"+ {len([r for r in dataset_build_log if r['split']=='holdout'])} holdout")
pd.DataFrame(dataset_build_log)[["bird", "split", "nframes"]].to_string(index=False) \
    and print(pd.DataFrame(dataset_build_log)[["bird", "split", "nframes"]].to_string(index=False))

INFO | Frame selection seed set to 42


[DavidBowie] Building training dataset (n=200 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[DavidBowie] Building training dataset (n=400 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...

...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (n=400 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (n=800 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO | Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (n=1400 frames, seed=42) ...
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...
...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset

────────────────────────────────────────────────────────────
Dataset builds complete: 8 training  + 0 holdout
      bird split  nframes
DavidBowie train      200
DavidBowie train      400
DavidBowie train      800
DavidBowie train     1400
    Endive train      200
    Endive train      400
    Endive train      800
    Endive train     1400


In [8]:
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
dataset_frame_counts = []

for (bird_name, split_key), config_path in sorted(
    combined_configs.items(),
    key=lambda item: (item[0][0], str(item[0][1])),
):
    labeled_dir = Path(config_path).parent / "labeled-data"
    image_paths = [
        p for p in labeled_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    unique_image_files = {
        p.relative_to(labeled_dir).as_posix()
        for p in image_paths
    }

    # Collapse cam1/cam2 pairs to one logical frame ID when possible.
    unique_logical_frames = {
        p.stem.replace("_cam1_", "_cam_").replace("_cam2_", "_cam_")
        for p in image_paths
    }

    dataset_frame_counts.append({
        "bird": bird_name,
        "split": "holdout" if split_key == "holdout" else "train",
        "nframes": HOLDOUT_N if split_key == "holdout" else split_key,
        "unique_image_files": len(unique_image_files),
        "unique_logical_frames": len(unique_logical_frames),
        "labeled_data_dir": str(labeled_dir),
    })

dataset_frame_counts_df = pd.DataFrame(dataset_frame_counts).sort_values(
    ["bird", "split", "nframes"]
)

print(
    dataset_frame_counts_df[
        ["bird", "split", "nframes", "unique_image_files", "unique_logical_frames"]
    ].to_string(index=False)
)

dataset_frame_counts_df

      bird split  nframes  unique_image_files  unique_logical_frames
DavidBowie train      200                 400                    200
DavidBowie train      400                 800                    400
DavidBowie train      800                1600                    800
DavidBowie train     1400                2800                   1400
    Endive train      200                 400                    200
    Endive train      400                 800                    400
    Endive train      800                1600                    800
    Endive train     1400                2800                   1400


,bird,split,nframes,unique_image_files,unique_logical_frames,labeled_data_dir
1,DavidBowie,train,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
2,DavidBowie,train,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
3,DavidBowie,train,800,1600,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
0,DavidBowie,train,1400,2800,1400,C:\Users\Salle-Cineradio\Documents\MachineLear...
5,Endive,train,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
6,Endive,train,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
7,Endive,train,800,1600,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
4,Endive,train,1400,2800,1400,C:\Users\Salle-Cineradio\Documents\MachineLear...


In [9]:
[element for element in unique_image_files if element.__contains__('001')]

['Canari/Endive_20251606_F32_cam1_0016.png',
 'Canari/Endive_20251606_F32_cam2_0013.png',
 'Canari/Endive_20251606_F32_cam2_0001.png',
 'Canari/Endive_20251606_F32_cam1_0010.png',
 'Canari/Endive_20251606_F32_cam2_0010.png',
 'Canari/Endive_20251606_F32_cam1_0018.png',
 'Canari/Endive_20251606_F32_cam1_0001.png',
 'Canari/Endive_20251606_F32_cam2_0018.png',
 'Canari/Endive_20251606_F32_cam1_0013.png',
 'Canari/Endive_20251606_F32_cam2_0016.png']

In [10]:
unique_image_files

{'Canari/Endive_20251606_F32_cam1_3238.png',
 'Canari/Endive_20251606_F32_cam2_0156.png',
 'Canari/Endive_20251606_F32_cam2_1465.png',
 'Canari/Endive_20251606_F32_cam2_2053.png',
 'Canari/Endive_20251606_F32_cam2_2837.png',
 'Canari/Endive_20251606_F32_cam2_3665.png',
 'Canari/Endive_20251606_F32_cam1_1754.png',
 'Canari/Endive_20251606_F32_cam2_1083.png',
 'Canari/Endive_20251606_F32_cam1_1005.png',
 'Canari/Endive_20251606_F32_cam2_2483.png',
 'Canari/Endive_20251606_F32_cam1_1542.png',
 'Canari/Endive_20251606_F32_cam2_3158.png',
 'Canari/Endive_20251606_F32_cam2_1038.png',
 'Canari/Endive_20251606_F32_cam1_1048.png',
 'Canari/Endive_20251606_F32_cam2_1678.png',
 'Canari/Endive_20251606_F32_cam1_1901.png',
 'Canari/Endive_20251606_F32_cam2_3082.png',
 'Canari/Endive_20251606_F32_cam1_0579.png',
 'Canari/Endive_20251606_F32_cam1_2701.png',
 'Canari/Endive_20251606_F32_cam1_1076.png',
 'Canari/Endive_20251606_F32_cam2_3375.png',
 'Canari/Endive_20251606_F32_cam2_3353.png',
 'Canari/E

## 7. Training — ResNet50 vs HRNet

Iterates over all **12 combinations** (2 architectures × 3 frame counts × 2 birds).  
Architecture is hot-swapped by patching `net_type` in each project's `config.yaml` before calling `create_training_dataset`.  
Wall-clock training time is recorded per run for comparison.

In [11]:

def set_net_type(config_path: Path, net_type: str) -> None:
    """Patch the net_type field in a DLC config.yaml in-place."""
    config_path = Path(config_path)
    with open(config_path, "r", encoding="utf-8") as fh:
        cfg = yaml.safe_load(fh)
    cfg["default_net_type"] = net_type
    with open(config_path, "w", encoding="utf-8") as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)

RUN_TRAINING      = True
USE_FULL_EPOCHS   = True
epochs = EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE
training_results = []  # list of dicts — one entry per (bird, n, arch) run

if not RUN_TRAINING:
    print("DRY RUN — set RUN_TRAINING = True to execute training.")
    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            for arch in ARCHITECTURES:
                training_results.append({
                    "bird": bird_name, "nframes": n, "architecture": arch,
                    "epochs": epochs, "trained": False,
                    "train_time_s": None, "latest_snapshot": None,
                    "notes": "dry run",
                })
else:
    for bird_name in BIRDS:
        for n in FRAME_COUNTS:
            train_cfg = combined_configs[(bird_name, n)]
            for arch in ARCHITECTURES:
                run_label = f"{bird_name} | n={n} | {arch}"
                print(f"\n{'─'*60}")
                print(f"Training: {run_label}")

                # Switch architecture in config.yaml
                set_net_type(train_cfg, arch)

                # Time the full create_training_dataset + train_network cycle
                t0 = time.perf_counter()
                try:
                    latest_snap = dlcs.create_and_train(
                        config_path   = train_cfg,
                        epochs        = epochs,
                        snapshot_path = None,   # always start fresh per variant
                    )
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": True,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": latest_snap,
                        "notes": "",
                    })
                    print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                except Exception as exc:
                    elapsed = time.perf_counter() - t0
                    training_results.append({
                        "bird": bird_name, "nframes": n, "architecture": arch,
                        "epochs": epochs, "trained": False,
                        "train_time_s": round(elapsed, 1),
                        "latest_snapshot": None,
                        "notes": str(exc),
                    })
                    print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")
pd.DataFrame(training_results)[["bird", "nframes", "architecture", "trained", "train_time_s"]]

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: DavidBowie | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\labeled-data\DB17_cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_combined_n200\\Canari-Tyler-2026-04-08\\labeled-data\\Canari', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_combined_n200\\Canari-Tyler-2026-04-08\\labeled-data\\DB17_cam1']
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08
  pose_config_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\CanariApr8-trainset95shuffle1\train\pytorch_config.yaml
  bo

  ERROR: No snapshots found under pattern: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n200\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\*\train\snapshots\snapshot-*.pt

────────────────────────────────────────────────────────────
Training: DavidBowie | n=400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\labeled-data\DB17_cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_combined_n400\\Canari-Tyler-2026-04-08\\labeled-data\\Canari', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08
  pose_config_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\TestData\Canari_combined_n400\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\CanariApr8-trainset95shuffle1\train\pytorch_config.yaml
  bo

...

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08
  pose_config_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\CanariApr8-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts:

  ERROR: No snapshots found under pattern: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n800\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\*\train\snapshots\snapshot-*.pt

────────────────────────────────────────────────────────────
Training: Endive | n=1400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\labeled-data\EndiveF42_20251606_Cam1_short\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\Endive\\TestData\\Canari_combined_n1400\\Canari-Tyler-2026-04-08\\labeled-data\\Canari', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSon

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08
  pose_config_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\CanariApr8-trainset95shuffle1\train\pytorch_config.yaml
  bodypart

  ERROR: No snapshots found under pattern: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\Endive\TestData\Canari_combined_n1400\Canari-Tyler-2026-04-08\dlc-models-pytorch\iteration-0\*\train\snapshots\snapshot-*.pt

────────────────────────────────────────────────────────────
Training runs recorded: 8


,bird,nframes,architecture,trained,train_time_s
0,DavidBowie,200,resnet_50,False,3648.2
1,DavidBowie,400,resnet_50,False,7162.0
2,DavidBowie,800,resnet_50,False,14416.9
3,DavidBowie,1400,resnet_50,False,25569.8
4,Endive,200,resnet_50,False,3649.7
5,Endive,400,resnet_50,False,8491.6
6,Endive,800,resnet_50,False,14469.8
7,Endive,1400,resnet_50,False,25427.9


### Model Manipulation

How are we able to pull and train specific layers of the model to improve/transfer